In [1]:
# %%bash
# SITE=$(python -c "import site; print(site.getsitepackages()[0])")
# rm -rf "$SITE/triton" "$SITE/bitsandbytes"

# # 0.45.0+ ships precompiled cuda124 binaries
# pip install -q --no-cache-dir "bitsandbytes==0.45.0"

# pip install -q --no-cache-dir \
#     transformers==4.44.0 \
#     peft==0.12.0 \
#     accelerate==0.33.0 \
#     trl==0.9.6 \
#     datasets==2.21.0 \
#     rouge-score==0.1.2 \
#     bert-score==0.3.13 \
#     sacrebleu==2.4.0 \
#     sentencepiece==0.1.99 \
#     pandas scikit-learn matplotlib seaborn

# # Verify
# python -c "
# import importlib.util
# print('triton present:', importlib.util.find_spec('triton') is not None)
# import bitsandbytes as bnb
# import importlib.metadata
# print('bitsandbytes', importlib.metadata.version('bitsandbytes'), 'loaded OK')
# import torch
# print('CUDA available:', torch.cuda.is_available())
# "
# echo "DONE. Now restart the kernel."

In [2]:
# import importlib.metadata, subprocess, sys

# expected = {
#     "bitsandbytes": "0.45.0",
#     "transformers":  "4.44.0",
#     "peft":          "0.12.0",
#     "accelerate":    "0.33.0",
#     "trl":           "0.9.6",
# }

# all_ok = True
# for pkg, ver in expected.items():
#     result = subprocess.check_output([sys.executable, "-m", "pip", "show", pkg]).decode()
#     installed = [l.split(": ")[1] for l in result.splitlines() if l.startswith("Version")][0]
#     status = "✅" if installed == ver else "❌ MISMATCH"
#     if installed != ver:
#         all_ok = False
#     print(f"{status}  {pkg}: installed={installed}, expected={ver}")

# import importlib.util
# triton_ok = importlib.util.find_spec("triton") is None
# print(f"{'✅' if triton_ok else '❌'} triton absent: {triton_ok}  ← must be True")

# if all_ok and triton_ok:
#     print("\n✅ All versions correct. Safe to proceed.")
# else:
#     print("\n❌ Fix mismatches above before continuing.")

In [3]:
import os, json, random, warnings
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, MBartForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, get_cosine_schedule_with_warmup,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from rouge_score import rouge_scorer as rouge_lib
from bert_score import score as bert_score_fn
from datasets import Dataset as HFDataset
import sacrebleu

warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"]   = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ── Global Config ─────────────────────────────────────────────
SEED       = 42
DATA_PATH  = "good_rows_maithili_1.csv"
MODEL_NAME = "ai4bharat/IndicBART"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

SRC_TAG    = "<2hi>"
TGT_TAG    = "<2hi>"

/mnt/Data/yashv7523/anaconda3/envs/ame=Deepak/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["article_text", "summary", "article_headline"])
df["article_text"] = df["article_text"].astype(str).str.strip()
df["summary"]      = df["summary"].astype(str).str.strip()
df["headline"]     = df["article_headline"].astype(str).str.strip()
df = df[df["article_text"].ne("") & df["summary"].ne("") & df["headline"].ne("")]
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Total samples: {len(df)}")

Total samples: 2010


In [5]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")


Train: 1608, Val: 201, Test: 201


In [6]:
print("\nLoading IndicBART tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=False)
tgt_tag_id = tokenizer.convert_tokens_to_ids(TGT_TAG)
assert tgt_tag_id != tokenizer.unk_token_id, "Tag not in vocab!"
print(f"Vocab size: {tokenizer.vocab_size}, TGT tag id: {tgt_tag_id}")



Loading IndicBART tokenizer...
Vocab size: 64000, TGT tag id: 64006


In [7]:
def word_tok(text):
    return text.strip()

In [11]:
def run_task(task_name, input_col, target_col,
             max_input_len, max_target_len, max_new_tokens,
             num_epochs, lr, batch_size, grad_accum,
             output_dir, final_dir, results_dir):

    for d in [output_dir, final_dir, results_dir]:
        os.makedirs(d, exist_ok=True)

    def tokenize(texts, targets):
        inputs_tagged = [f"{SRC_TAG} {t}" for t in texts]
        enc_inputs = tokenizer(inputs_tagged, max_length=max_input_len,
                               truncation=True, padding="max_length", return_tensors="pt")
        enc_targets = tokenizer(targets, max_length=max_target_len,
                                truncation=True, padding="max_length", return_tensors="pt")
        labels = enc_targets["input_ids"].clone()
        labels[labels == tokenizer.pad_token_id] = -100
        return enc_inputs["input_ids"], enc_inputs["attention_mask"], labels

    print(f"\n[{task_name}] Tokenizing...")
    train_inp, train_mask, train_lbl = tokenize(train_df[input_col].tolist(), train_df[target_col].tolist())
    val_inp,   val_mask,   val_lbl   = tokenize(val_df[input_col].tolist(),   val_df[target_col].tolist())

    n_valid = (train_lbl[0] != -100).sum().item()
    print(f"Sample 0: label_len={train_lbl.shape[1]}, valid={n_valid}")
    print(f"Label preview: {tokenizer.decode(train_lbl[0][train_lbl[0]!=-100][:20].tolist())}")
    assert n_valid > 0, "Labels all masked!"

    class Seq2SeqDS(torch.utils.data.Dataset):
        def __init__(self, inp, mask, lbl):
            self.inp = inp; self.mask = mask; self.lbl = lbl
        def __len__(self): return len(self.inp)
        def __getitem__(self, i):
            return {"input_ids": self.inp[i], "attention_mask": self.mask[i], "labels": self.lbl[i]}

    train_ds = Seq2SeqDS(train_inp, train_mask, train_lbl)
    val_ds   = Seq2SeqDS(val_inp,   val_mask,   val_lbl)

    print(f"\n[{task_name}] Loading model...")
    model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)
    model.config.pad_token_id           = tokenizer.pad_token_id
    model.config.decoder_start_token_id = tgt_tag_id
    model.config.forced_eos_token_id    = tokenizer.eos_token_id
    model = model.cuda()
    print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

    model.eval()
    with torch.no_grad():
        test_out = model(input_ids=train_inp[:2].cuda(),
                         attention_mask=train_mask[:2].cuda(),
                         labels=train_lbl[:2].cuda())
    print(f"Pre-training loss: {test_out.loss.item():.4f}")
    assert test_out.loss.item() > 0, "Loss is 0 — check setup!"

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    optimizer    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps  = (len(train_loader) // grad_accum) * num_epochs
    warmup_steps = int(0.1 * total_steps)
    scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    best_val_loss  = float("inf")
    patience       = 3
    patience_count = 0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader):
            out  = model(input_ids=batch["input_ids"].cuda(),
                         attention_mask=batch["attention_mask"].cuda(),
                         labels=batch["labels"].cuda())
            loss = out.loss / grad_accum
            loss.backward()
            total_loss += out.loss.item()
            if (step + 1) % grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

        avg_train = total_loss / len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                out = model(input_ids=batch["input_ids"].cuda(),
                            attention_mask=batch["attention_mask"].cuda(),
                            labels=batch["labels"].cuda())
                val_loss += out.loss.item()
        avg_val = val_loss / len(val_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | Train: {avg_train:.4f} | Val: {avg_val:.4f}")

        if avg_val < best_val_loss:
            best_val_loss  = avg_val
            patience_count = 0
            model.save_pretrained(final_dir)
            tokenizer.save_pretrained(final_dir)
            print(f"  ✅ Saved (val_loss={best_val_loss:.4f})")
        else:
            patience_count += 1
            print(f"  No improvement ({patience_count}/{patience})")
            if patience_count >= patience:
                print("  Early stopping."); break

    del model; torch.cuda.empty_cache()

    # ── Inference ──────────────────────────────────────────────
    print(f"\n[{task_name}] Loading best model for inference...")
    ft_model = MBartForConditionalGeneration.from_pretrained(final_dir)
    ft_model.config.pad_token_id           = tokenizer.pad_token_id
    ft_model.config.decoder_start_token_id = tgt_tag_id
    ft_model.config.forced_eos_token_id    = tokenizer.eos_token_id
    ft_model = ft_model.cuda().eval()

    def generate(text):
        inputs = tokenizer(f"{SRC_TAG} {text}", return_tensors="pt",
                           max_length=max_input_len, truncation=True, padding=False).to(ft_model.device)
        inputs.pop("token_type_ids", None)
        with torch.no_grad():
            out = ft_model.generate(**inputs, max_new_tokens=max_new_tokens,
                                    min_new_tokens=max(5, max_new_tokens // 5),
                                    num_beams=4, length_penalty=1.0,
                                    no_repeat_ngram_size=3, early_stopping=True,
                                    decoder_start_token_id=tgt_tag_id,
                                    forced_eos_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(out[0], skip_special_tokens=True).strip()

    print(f"\n[{task_name}] Inference on {len(test_df)} samples...")
    preds, golds = [], []
    for i, (_, row) in enumerate(test_df.iterrows()):
        preds.append(generate(row[input_col]))
        golds.append(row[target_col])
        if (i + 1) % 20 == 0:
            print(f"  [{i+1}/{len(test_df)}]")

    pred_df = test_df.copy()
    pred_df[f"generated_{target_col}"] = preds
    pred_df.to_csv(f"{results_dir}/predictions.csv", index=False)

    # ── Evaluation ─────────────────────────────────────────────
    # ── Manual ROUGE for Devanagari ────────────────────────────────
    def tokenize_words(text):
        return text.strip().split()

    def get_ngrams(tokens, n):
        return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

    def compute_rouge_n(gold_tokens, pred_tokens, n):
        gold_ngrams = get_ngrams(gold_tokens, n)
        pred_ngrams = get_ngrams(pred_tokens, n)
        if not gold_ngrams or not pred_ngrams:
            return 0.0, 0.0, 0.0
        gold_set = {}
        for ng in gold_ngrams:
            gold_set[ng] = gold_set.get(ng, 0) + 1
        pred_set = {}
        for ng in pred_ngrams:
            pred_set[ng] = pred_set.get(ng, 0) + 1
        overlap = sum(min(gold_set.get(ng, 0), pred_set.get(ng, 0))
                      for ng in pred_set)
        precision = overlap / len(pred_ngrams) if pred_ngrams else 0.0
        recall    = overlap / len(gold_ngrams) if gold_ngrams else 0.0
        f1 = (2 * precision * recall / (precision + recall)
              if (precision + recall) > 0 else 0.0)
        return precision, recall, f1

    def compute_rouge_l(gold_tokens, pred_tokens):
        m, n = len(gold_tokens), len(pred_tokens)
        if m == 0 or n == 0:
            return 0.0, 0.0, 0.0
        # LCS dynamic programming
        dp = [[0]*(n+1) for _ in range(m+1)]
        for i in range(1, m+1):
            for j in range(1, n+1):
                if gold_tokens[i-1] == pred_tokens[j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                else:
                    dp[i][j] = max(dp[i-1][j], dp[i][j-1])
        lcs = dp[m][n]
        precision = lcs / n if n > 0 else 0.0
        recall    = lcs / m if m > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)
              if (precision + recall) > 0 else 0.0)
        return precision, recall, f1

    def compute_bleu_word(references, predictions):
        """Manual word-level BLEU using sacrebleu with 13a tokenizer"""
        return sacrebleu.corpus_bleu(predictions, [references], tokenize="13a")

    # ── Run Evaluation ─────────────────────────────────────────────
    r1_p_l, r1_r_l, r1_f = [], [], []
    r2_p_l, r2_r_l, r2_f = [], [], []
    rl_p_l, rl_r_l, rl_f = [], [], []
    per_records = []
    empty_count = 0

    print(f"\n{'='*70}\n[{task_name}] PER-INSTANCE ROUGE (word-level manual)\n{'='*70}")

    for idx, (gold, pred) in enumerate(zip(golds, preds)):
        ep = pred if pred.strip() else "खाली"
        if not pred.strip():
            empty_count += 1

        gold_tokens = tokenize_words(gold)
        pred_tokens = tokenize_words(ep)

        r1_p, r1_r, r1_f1 = compute_rouge_n(gold_tokens, pred_tokens, 1)
        r2_p, r2_r, r2_f1 = compute_rouge_n(gold_tokens, pred_tokens, 2)
        rl_p, rl_r, rl_f1 = compute_rouge_l(gold_tokens, pred_tokens)

        r1_p_l.append(r1_p); r1_r_l.append(r1_r); r1_f.append(r1_f1)
        r2_p_l.append(r2_p); r2_r_l.append(r2_r); r2_f.append(r2_f1)
        rl_p_l.append(rl_p); rl_r_l.append(rl_r); rl_f.append(rl_f1)

        per_records.append({
            "index"      : idx,
            "gold"       : gold,
            "pred"       : pred,
            "ROUGE-1_F1" : round(r1_f1, 4),
            "ROUGE-2_F1" : round(r2_f1, 4),
            "ROUGE-L_F1" : round(rl_f1, 4),
        })

        if idx < 10:
            print(f"\nSample {idx+1}")
            print(f"  GOLD : {gold[:120]}")
            print(f"  PRED : {pred[:120]}")
            print(f"  R1={r1_f1:.4f} R2={r2_f1:.4f} RL={rl_f1:.4f}")

    if empty_count:
        print(f"\n⚠️ {empty_count} empty predictions")

    pd.DataFrame(per_records).to_csv(
        f"{results_dir}/per_instance_rouge.csv",
        index=False, encoding="utf-8-sig")

    # BERTScore
    print(f"\n[{task_name}] Computing BERTScore...")
    P, R, F = bert_score_fn(
        preds, golds, lang="hi",
        model_type="bert-base-multilingual-cased",
        batch_size=8, verbose=False)

    # BLEU word-level
    print(f"[{task_name}] Computing BLEU...")
    bleu = sacrebleu.corpus_bleu(preds, [golds], tokenize="13a")

    print(f"\n{'='*70}\n[{task_name}] OVERALL METRICS\n{'='*70}")
    print(f"{'Metric':<12} {'P':>8} {'R':>8} {'F1':>8}")
    print(f"{'ROUGE-1':<12} {np.mean(r1_p_l):>8.4f} {np.mean(r1_r_l):>8.4f} {np.mean(r1_f):>8.4f}")
    print(f"{'ROUGE-2':<12} {np.mean(r2_p_l):>8.4f} {np.mean(r2_r_l):>8.4f} {np.mean(r2_f):>8.4f}")
    print(f"{'ROUGE-L':<12} {np.mean(rl_p_l):>8.4f} {np.mean(rl_r_l):>8.4f} {np.mean(rl_f):>8.4f}")
    print(f"{'BERTScore':<12} {P.mean().item():>8.4f} {R.mean().item():>8.4f} {F.mean().item():>8.4f}")
    print(f"\nBLEU (word)  {bleu.score:.4f}")
    print(f"BLEU-1       {bleu.precisions[0]:.4f}")
    print(f"BLEU-2       {bleu.precisions[1]:.4f}")
    print(f"BLEU-3       {bleu.precisions[2]:.4f}")
    print(f"BLEU-4       {bleu.precisions[3]:.4f}")
    print(f"BP           {bleu.bp:.4f}")

    metrics = {
        "ROUGE-1"  : {"P": float(np.mean(r1_p_l)), "R": float(np.mean(r1_r_l)), "F1": float(np.mean(r1_f))},
        "ROUGE-2"  : {"P": float(np.mean(r2_p_l)), "R": float(np.mean(r2_r_l)), "F1": float(np.mean(r2_f))},
        "ROUGE-L"  : {"P": float(np.mean(rl_p_l)), "R": float(np.mean(rl_r_l)), "F1": float(np.mean(rl_f))},
        "BERTScore": {"P": float(P.mean().item()),  "R": float(R.mean().item()),  "F1": float(F.mean().item())},
        "BLEU"     : {"score" : float(bleu.score),
                      "BLEU-1": float(bleu.precisions[0]),
                      "BLEU-2": float(bleu.precisions[1]),
                      "BLEU-3": float(bleu.precisions[2]),
                      "BLEU-4": float(bleu.precisions[3]),
                      "BP"    : float(bleu.bp)},
    }
    with open(f"{results_dir}/metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print(f"\nMetrics saved to {results_dir}/metrics.json")

    del ft_model; torch.cuda.empty_cache()
    return metrics

In [12]:
# ── Run both tasks ─────────────────────────────────────────────
sum_metrics = run_task(
    task_name="SUMMARIZATION", input_col="article_text", target_col="summary",
    max_input_len=512, max_target_len=200, max_new_tokens=200,
    num_epochs=25, lr=3e-5, batch_size=8, grad_accum=2,
    output_dir="./indicbart_maithil_sum_checkpoints",
    final_dir="./indicbart_maithili_sum_final",
    results_dir="./indicbart_maithili_sum_results",
)


[SUMMARIZATION] Tokenizing...
Sample 0: label_len=200, valid=117
Label preview: [CLS]दहेज प्रथा पर कविता में भारती झा द्वारा ई प्रथा के निन्दा कयल गेल अछि, जे क

[SUMMARIZATION] Loading model...
Params: 244,017,152
Pre-training loss: 6.0790
Epoch 1/25 | Train: 4.7012 | Val: 3.7146
  ✅ Saved (val_loss=3.7146)
Epoch 2/25 | Train: 3.5899 | Val: 3.2316
  ✅ Saved (val_loss=3.2316)
Epoch 3/25 | Train: 3.2213 | Val: 2.9956
  ✅ Saved (val_loss=2.9956)
Epoch 4/25 | Train: 3.0298 | Val: 2.8495
  ✅ Saved (val_loss=2.8495)
Epoch 5/25 | Train: 2.9070 | Val: 2.7566
  ✅ Saved (val_loss=2.7566)
Epoch 6/25 | Train: 2.8093 | Val: 2.7068
  ✅ Saved (val_loss=2.7068)
Epoch 7/25 | Train: 2.7431 | Val: 2.6294
  ✅ Saved (val_loss=2.6294)
Epoch 8/25 | Train: 2.6819 | Val: 2.5865
  ✅ Saved (val_loss=2.5865)
Epoch 9/25 | Train: 2.6255 | Val: 2.5463
  ✅ Saved (val_loss=2.5463)
Epoch 10/25 | Train: 2.5819 | Val: 2.5121
  ✅ Saved (val_loss=2.5121)
Epoch 11/25 | Train: 2.5322 | Val: 2.4731
  ✅ Saved (val_loss=2.473

In [13]:
head_metrics = run_task(
    task_name="HEADLINE", input_col="article_text", target_col="headline",
    max_input_len=512, max_target_len=64, max_new_tokens=64,
    num_epochs=25, lr=3e-5, batch_size=8, grad_accum=2,
    output_dir="./indicbart_maithili_head_checkpoints",
    final_dir="./indicbart_maithili_head_final",
    results_dir="./indicbart_maithili_head_results",
)


[HEADLINE] Tokenizing...
Sample 0: label_len=64, valid=22
Label preview: [CLS]भारती झा केर दुइ गोट रचना – दहेज प्रथा आ कतय गेल मिथिलाक दल

[HEADLINE] Loading model...
Params: 244,017,152
Pre-training loss: 6.2299
Epoch 1/25 | Train: 6.0479 | Val: 5.0678
  ✅ Saved (val_loss=5.0678)
Epoch 2/25 | Train: 5.2205 | Val: 4.7778
  ✅ Saved (val_loss=4.7778)
Epoch 3/25 | Train: 4.9569 | Val: 4.5483
  ✅ Saved (val_loss=4.5483)
Epoch 4/25 | Train: 4.7421 | Val: 4.3881
  ✅ Saved (val_loss=4.3881)
Epoch 5/25 | Train: 4.5420 | Val: 4.2219
  ✅ Saved (val_loss=4.2219)
Epoch 6/25 | Train: 4.3770 | Val: 4.0724
  ✅ Saved (val_loss=4.0724)
Epoch 7/25 | Train: 4.2266 | Val: 3.9372
  ✅ Saved (val_loss=3.9372)
Epoch 8/25 | Train: 4.0822 | Val: 3.7998
  ✅ Saved (val_loss=3.7998)
Epoch 9/25 | Train: 3.9294 | Val: 3.7106
  ✅ Saved (val_loss=3.7106)
Epoch 10/25 | Train: 3.7975 | Val: 3.6313
  ✅ Saved (val_loss=3.6313)
Epoch 11/25 | Train: 3.7080 | Val: 3.5749
  ✅ Saved (val_loss=3.5749)
Epoch 12/25 | Train: 3.61

In [14]:
print("\n" + "="*70)
print("FINAL SUMMARY — MAITHILI")
print("="*70)
print(f"Summarization  ROUGE-1 F1 : {sum_metrics['ROUGE-1']['F1']:.4f}")
print(f"               ROUGE-2 F1 : {sum_metrics['ROUGE-2']['F1']:.4f}")
print(f"               BERTScore  : {sum_metrics['BERTScore']['F1']:.4f}")
print(f"               BLEU       : {sum_metrics['BLEU']['score']:.4f}")
print(f"               BLEU-1     : {sum_metrics['BLEU']['BLEU-1']:.4f}")
print(f"               BLEU-2     : {sum_metrics['BLEU']['BLEU-2']:.4f}")
print(f"               BLEU-3     : {sum_metrics['BLEU']['BLEU-3']:.4f}")
print(f"               BLEU-4     : {sum_metrics['BLEU']['BLEU-4']:.4f}")
print(f"Headline       ROUGE-1 F1 : {head_metrics['ROUGE-1']['F1']:.4f}")
print(f"               ROUGE-2 F1 : {head_metrics['ROUGE-2']['F1']:.4f}")
print(f"               BERTScore  : {head_metrics['BERTScore']['F1']:.4f}")
print(f"               BLEU       : {head_metrics['BLEU']['score']:.4f}")
print(f"               BLEU-1     : {head_metrics['BLEU']['BLEU-1']:.4f}")
print(f"               BLEU-2     : {head_metrics['BLEU']['BLEU-2']:.4f}")
print(f"               BLEU-3     : {head_metrics['BLEU']['BLEU-3']:.4f}")
print(f"               BLEU-4     : {head_metrics['BLEU']['BLEU-4']:.4f}")


FINAL SUMMARY — MAITHILI
Summarization  ROUGE-1 F1 : 0.3778
               ROUGE-2 F1 : 0.1567
               BERTScore  : 0.7414
               BLEU       : 11.5159
               BLEU-1     : 33.4452
               BLEU-2     : 14.2232
               BLEU-3     : 7.7451
               BLEU-4     : 4.7735
Headline       ROUGE-1 F1 : 0.1566
               ROUGE-2 F1 : 0.0614
               BERTScore  : 0.6950
               BLEU       : 3.2751
               BLEU-1     : 11.1850
               BLEU-2     : 4.3962
               BLEU-3     : 2.1600
               BLEU-4     : 1.0832
